In [1]:
import duckdb
from pathlib import Path

# Paths relative to notebooks/
input_file = "../brk_metrics.parquet"
output_dir = Path("../data/processed/pivot_by_year")
db_path = "../data/processed/brk_work.duckdb"
temp_dir = "../data/processed/duckdb_temp"

output_dir.mkdir(parents=True, exist_ok=True)
Path(temp_dir).mkdir(parents=True, exist_ok=True)

con = duckdb.connect(db_path)

con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")
con.execute("SET memory_limit='5GB'")
con.execute(f"SET temp_directory='{temp_dir}'")

years = con.execute(f"""
SELECT DISTINCT year(day_utc)::INT AS yr
FROM read_parquet('{input_file}')
ORDER BY yr
""").fetchall()

print("Years found:", years)

for (yr,) in years:
    print(f"Processing year {yr}")

    output_file = output_dir / f"brk_pivot_{yr}.parquet"

    con.execute(f"""
    COPY (
        PIVOT (
            SELECT day_utc, metric, value
            FROM read_parquet('{input_file}')
            WHERE year(day_utc) = {yr}
        )
        ON metric
        USING first(value)
        GROUP BY day_utc
    )
    TO '{output_file.as_posix()}'
    (FORMAT PARQUET)
    """)

con.close()

Years found: [(2009,), (2010,), (2011,), (2012,), (2013,), (2014,), (2015,), (2016,), (2017,), (2018,), (2019,), (2020,), (2021,), (2022,), (2023,), (2024,), (2025,), (2026,)]
Processing year 2009
Processing year 2010
Processing year 2011
Processing year 2012
Processing year 2013
Processing year 2014
Processing year 2015
Processing year 2016
Processing year 2017
Processing year 2018
Processing year 2019
Processing year 2020
Processing year 2021
Processing year 2022
Processing year 2023
Processing year 2024
Processing year 2025
Processing year 2026
